# 🔎 Exploratory Data Analysis — Superstore Sales
**Stack:** Pandas, NumPy

In [ ]:
import numpy as np, pandas as pd
pd.set_option('display.max_columns', None)
df = pd.read_csv('../Dataset/Cleaned_Data.csv', parse_dates=['order_date','ship_date'])
print(df.shape)
df.head()

## 1. Descriptive Statistics

In [ ]:
df[['sales','quantity','discount','profit','profit_margin_pct','shipping_days']].describe().T

In [ ]:
print('Orders        :', df.order_id.nunique())
print('Customers     :', df.customer_id.nunique())
print('Products      :', df.product_id.nunique())
print('Date range    :', df.order_date.min().date(),'->',df.order_date.max().date())
print('Total Sales   : $%.2f'%df.sales.sum())
print('Total Profit  : $%.2f'%df.profit.sum())
print('Avg Margin    : %.2f%%'%df.profit_margin_pct.mean())
print('Avg Order Val : $%.2f'%(df.sales.sum()/df.order_id.nunique()))

## 2. Correlation Analysis

In [ ]:
num = ['sales','quantity','discount','profit','profit_margin_pct','unit_price','shipping_days']
df[num].corr().round(2)

**Insight:** Discount is negatively correlated with profit margin — heavy discounting erodes profitability.

## 3. Trend Analysis (monthly)

In [ ]:
trend = df.groupby('order_year_month').agg(sales=('sales','sum'),profit=('profit','sum'),
        orders=('order_id','nunique')).reset_index()
trend.tail(12)

In [ ]:
yoy = df.groupby('order_year').agg(sales=('sales','sum'),profit=('profit','sum')).reset_index()
yoy['sales_growth_%']=yoy['sales'].pct_change().mul(100).round(1)
yoy['profit_growth_%']=yoy['profit'].pct_change().mul(100).round(1)
yoy

## 4. Category & Sub-Category Analysis

In [ ]:
df.groupby('category').agg(sales=('sales','sum'),profit=('profit','sum'),
    margin=('profit_margin_pct','mean'),orders=('order_id','nunique')).round(2)

In [ ]:
df.groupby('sub_category').agg(sales=('sales','sum'),profit=('profit','sum'))\
  .sort_values('profit').round(0).head(10)

**Insight:** Technology drives the highest margin; some Furniture sub-categories are loss-leaders.

## 5. Segment & Region Analysis

In [ ]:
df.groupby('segment').agg(sales=('sales','sum'),profit=('profit','sum'),
    customers=('customer_id','nunique')).round(2)

In [ ]:
df.groupby('region').agg(sales=('sales','sum'),profit=('profit','sum'),
    orders=('order_id','nunique')).round(2)

## 6. Segmentation — Top States & Customers

In [ ]:
df.groupby('state')['sales'].sum().sort_values(ascending=False).head(10).round(0)

In [ ]:
df.groupby('customer_name').agg(clv=('sales','sum'),orders=('order_id','nunique'))\
  .sort_values('clv',ascending=False).head(10).round(0)

## 7. Pattern Detection

In [ ]:
print('Sales by weekday:'); print(df.groupby('order_weekday')['sales'].sum().round(0))
print('\nShip mode share:'); print(df.ship_mode.value_counts(normalize=True).round(3))
print('\nLoss-making orders: %.1f%%'%(100*(df.profit<0).mean()))

## 📌 Key EDA Findings
1. Sales trend up YoY with strong Q4 seasonality.
2. Technology = top margin; Furniture = thin/negative margins in places.
3. Consumer segment dominates revenue; Home Office is smallest.
4. West & East lead regional sales.
5. High discounts strongly depress margins; a chunk of orders run at a loss.
6. Customer revenue is concentrated in a small top cohort (Pareto).